In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, concat_ws, lit, when, col

In [4]:

spark = SparkSession.builder \
    .appName("SilverLayer") \
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/05/07 16:39:43 WARN Utils: Your hostname, Saileshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/05/07 16:39:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/saileshpola/.ivy2/cache
The jars for the packages stored in: /Users/saileshpola/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3e3be718-e5d0-46f0-9cb0-7f7939fda3b6;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central


:: loading settings :: url = jar:file:/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 102ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-3e3be718-e5d0-46f0-9cb0-7f7939fda3b6
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
26/05/07 16:39:43 WARN NativeCodeLoader: Unable to load native-hado

In [6]:
df = spark.readStream.format("delta").load("../BronzeLayer/data/bronze/trades")

26/05/08 15:20:19 ERROR Inbox: Ignoring error                                   
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apach

In [ ]:
parsed_ts = to_timestamp(col("trade_timestamp"))

trade_id_valid = col("trade_id").isNotNull()
trader_id_valid = col("trader_id").isNotNull()
price_type_valid = col("price").cast("double").isNotNull()
price_value_valid = col("price") > 0
quantity_type_valid = col("quantity").cast("int").isNotNull()
quantity_value_valid = col("quantity") > 0
trade_timestamp_valid = parsed_ts.isNotNull()
trade_timestamp_not_future = parsed_ts <= col("ingestion_time")
side_valid = col("side").isin(["BUY", "SELL"])
exchange_valid = col("exchange").isin(["NYSE", "NASDAQ", "BINANCE"])


In [4]:
valid_condition = (
    trade_id_valid &
    trader_id_valid &
    price_type_valid &
    price_value_valid &
    quantity_type_valid &
    quantity_value_valid &
    trade_timestamp_valid &
    trade_timestamp_not_future &
    side_valid &
    exchange_valid
)

valid_trades = df.filter(valid_condition)
dlq_trades = (df.filter(~valid_condition).withColumn("dlq_reason", concat_ws(
        ", ",
        when(~trade_id_valid, lit("missing_trade_id")),
        when(~trader_id_valid, lit("missing_trader_id")),
        when(~price_type_valid, lit("invalid_price_type")),
        when(price_type_valid & ~price_value_valid, lit("invalid_price_value")),
        when(~quantity_type_valid, lit("invalid_quantity_type")),
        when(quantity_type_valid & ~quantity_value_valid, lit("invalid_quantity_value")),
        when(~trade_timestamp_valid, lit("invalid_trade_timestamp")),
        when(trade_timestamp_valid & ~trade_timestamp_not_future, lit("future_trade_timestamp")),
        when(~side_valid, lit("invalid_side")),
        when(~exchange_valid, lit("invalid_exchange"))
    ))
              )

valid_trades = valid_trades.withColumn(
    "trade_timestamp",
    parsed_ts
)

In [5]:
valid_trades = valid_trades.withWatermark("trade_timestamp", "10 minutes").dropDuplicates(["trade_id"])

In [6]:
valid_query = (valid_trades.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", 'checkpoints/silver/trades')
        .option("path", "data/silver/trades")
        .queryName("silver_trades")
        .start())

In [7]:
dlq_query = (dlq_trades.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", 'checkpoints/dlq/trades')
        .option("path", "data/dlq/trades")
        .queryName("dlq_trades")
        .start())